In [2]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

# Limpieza de los CIE para exportacion como .paquet

## CIE 9

Se filtra CIE-9 para que solo tenga los procedimientos que sean una vacunación para enfermedad respiratoria explicita

In [11]:
df_cie9 = pd.read_excel('../datos_grd/CIE-9 .xlsx')

palabras_clave_resp = ['GRIPE', 'TUBERCULOSIS', 'TOSFERINA', 'RESFRIADO']

es_vacuna = df_cie9['Descripción'].str.contains('VACUNACION', case=False, na=False)
es_respiratoria = df_cie9['Descripción'].str.contains('|'.join(palabras_clave_resp), case=False, na=False)

cie9_filtrado = df_cie9[es_vacuna & es_respiratoria].copy()

cie9_filtrado.to_parquet('CIE9_vacunas_respiratorias.parquet', index=False)

print(f"CIE-9: Se aislaron {len(cie9_filtrado)} procedimientos de vacunación.")
display(cie9_filtrado)

CIE-9: Se aislaron 5 procedimientos de vacunación.


,Código,Descripción,Categoría,Sección,Capítulo
4585,99.33,VACUNACION TUBERCULOSIS,99 OTROS PROCEDIMIENTOS NO QUIRURGICOS,16. (87 99) PROCEDIMIENTOS DIAGNOSTICOS Y TERA...,16. (87 99) PROCEDIMIENTOS DIAGNOSTICOS Y TERA...
4589,99.37,VACUNACION TOSFERINA,99 OTROS PROCEDIMIENTOS NO QUIRURGICOS,16. (87 99) PROCEDIMIENTOS DIAGNOSTICOS Y TERA...,16. (87 99) PROCEDIMIENTOS DIAGNOSTICOS Y TERA...
4591,99.39,VACUNACION DIFTERIA-TETANOS-TOSFERINA,99 OTROS PROCEDIMIENTOS NO QUIRURGICOS,16. (87 99) PROCEDIMIENTOS DIAGNOSTICOS Y TERA...,16. (87 99) PROCEDIMIENTOS DIAGNOSTICOS Y TERA...
4602,99.51,VACUNACION RESFRIADO COMUN,99 OTROS PROCEDIMIENTOS NO QUIRURGICOS,16. (87 99) PROCEDIMIENTOS DIAGNOSTICOS Y TERA...,16. (87 99) PROCEDIMIENTOS DIAGNOSTICOS Y TERA...
4603,99.52,VACUNACION GRIPE,99 OTROS PROCEDIMIENTOS NO QUIRURGICOS,16. (87 99) PROCEDIMIENTOS DIAGNOSTICOS Y TERA...,16. (87 99) PROCEDIMIENTOS DIAGNOSTICOS Y TERA...


## CIE 10

Se filtra CIE-10 para que solo tenga los diagnosticos que sean una enfermedad respiratoria

In [13]:
df_cie10 = pd.read_excel('../datos_grd/CIE-10.xlsx')

cie10_filtrado = df_cie10[df_cie10['Código'].astype(str).str.startswith('J', na=False)].copy()

cie10_filtrado.to_parquet('CIE10_enfermedades_respiratorias.parquet', index=False)

print(f"CIE-10: Se aislaron {len(cie10_filtrado)} diagnósticos respiratorios.")
display(cie10_filtrado.head())

CIE-10: Se aislaron 288 diagnósticos respiratorios.


,Versión,Código,Descripción,Categoría,Sección,Capítulo
4159,CIE-v2013,J00,Rinofaringitis aguda [resfriado común],J00 RINOFARINGITIS AGUDA [RESFRIADO COMÚN],J00-J06 INFECCIONES AGUDAS DEL TRACTO RESPIRA...,Cap.10 ENFERMEDADES DEL APARATO RESPIRATORIO ...
4160,CIE-v2013,J01,Sinusitis aguda,J01 SINUSITIS AGUDA,J00-J06 INFECCIONES AGUDAS DEL TRACTO RESPIRA...,Cap.10 ENFERMEDADES DEL APARATO RESPIRATORIO ...
4161,CIE-v2013,J01.0,Sinusitis maxilar aguda,J01 SINUSITIS AGUDA,J00-J06 INFECCIONES AGUDAS DEL TRACTO RESPIRA...,Cap.10 ENFERMEDADES DEL APARATO RESPIRATORIO ...
4162,CIE-v2013,J01.1,Sinusitis frontal aguda,J01 SINUSITIS AGUDA,J00-J06 INFECCIONES AGUDAS DEL TRACTO RESPIRA...,Cap.10 ENFERMEDADES DEL APARATO RESPIRATORIO ...
4163,CIE-v2013,J01.2,Sinusitis etmoidal aguda,J01 SINUSITIS AGUDA,J00-J06 INFECCIONES AGUDAS DEL TRACTO RESPIRA...,Cap.10 ENFERMEDADES DEL APARATO RESPIRATORIO ...


# Filtracion de los GRD por enfermedad respiratoria

Procesa todos los archivos `GRD_PUBLICO_YYYY.txt` disponibles y exporta solo los registros cuyo `DIAGNOSTICO1` comienza con `J`, correspondiente al capítulo de enfermedades respiratorias de CIE-10.

Las salidas quedan en `datos_grd/filtrados_ER/` como `GRD_PUBLICO_YYYY_ER.txt`, conservando el formato original delimitado por `|`.

## 2025

In [3]:
import pandas as pd

df_grd_2025 = pd.read_csv('../datos_grd/GRD_PUBLICO_2025.txt', sep='|', encoding='Latin-1', low_memory=False)

mascara_respiratoria = df_grd_2025['DIAGNOSTICO1'].astype(str).str.strip().str.upper().str.startswith('J')

grd_2025_filtrado = df_grd_2025[mascara_respiratoria].copy()

ruta_salida = '../datos_grd/GRD_PUBLICO_2025_ER.txt'
grd_2025_filtrado.to_csv(
    ruta_salida,
    sep='|',
    index=False,
    encoding='Latin-1',
    lineterminator='\n'
)

print(f"Total de registros originales: {len(df_grd_2025):,}")
print(f"Registros retenidos (DIAGNOSTICO1 empieza con 'J'): {len(grd_2025_filtrado):,}")
print(f"Archivo exportado: {ruta_salida}")

display(grd_2025_filtrado[['ID_BENEFICIARIO', 'DIAGNOSTICO1', 'FECHA_INGRESO', 'FECHAALTA']].head())

Total de registros originales: 1,136,077
Registros retenidos (DIAGNOSTICO1 empieza con 'J'): 94,161
Archivo exportado: ../datos_grd/GRD_PUBLICO_2025_ER.txt


,ID_BENEFICIARIO,DIAGNOSTICO1,FECHA_INGRESO,FECHAALTA
17,75936253,J15.8,16-10-2025,17-10-2025
33,68793942,J90,10-04-2025,18-04-2025
62,104312917,J20.8,11-09-2025,15-09-2025
73,102182984,J12.8,23-09-2025,24-09-2025
74,69611205,J96.00,19-09-2025,28-09-2025


## 2024

In [4]:
df_grd_2024 = pd.read_csv('../datos_grd/GRD_PUBLICO_2024.txt', sep='|', encoding='Latin-1', low_memory=False)

mascara_respiratoria = df_grd_2024['DIAGNOSTICO1'].astype(str).str.strip().str.upper().str.startswith('J')

grd_2024_filtrado = df_grd_2024[mascara_respiratoria].copy()

ruta_salida = '../datos_grd/GRD_PUBLICO_2024_ER.txt'
grd_2024_filtrado.to_csv(
    ruta_salida,
    sep='|',
    index=False,
    encoding='Latin-1',
    lineterminator='\n'
)

print(f"Total de registros originales: {len(df_grd_2024):,}")
print(f"Registros retenidos (DIAGNOSTICO1 empieza con 'J'): {len(grd_2024_filtrado):,}")
print(f"Archivo exportado: {ruta_salida}")

display(grd_2024_filtrado[['ID_BENEFICIARIO', 'DIAGNOSTICO1', 'FECHA_INGRESO', 'FECHAALTA']].head())

Total de registros originales: 1,085,813
Registros retenidos (DIAGNOSTICO1 empieza con 'J'): 88,764
Archivo exportado: ../datos_grd/GRD_PUBLICO_2024_ER.txt


,ID_BENEFICIARIO,DIAGNOSTICO1,FECHA_INGRESO,FECHAALTA
14,80028465,J35.3,2024-07-06,2024-07-06
17,100044868,J15.1,2024-09-30,2024-10-22
26,99171002,J20.9,2024-05-24,2024-05-26
27,80270505,J35.3,2024-08-05,2024-08-05
33,70496766,J96.09,2024-10-04,2024-10-05


## 2023

In [ ]:
df_grd_2023 = pd.read_csv('../datos_grd/GRD_PUBLICO_2023.txt', sep='|', encoding='utf-16', low_memory=False)

mascara_respiratoria = df_grd_2023['DIAGNOSTICO1'].astype(str).str.strip().str.upper().str.startswith('J')

grd_2023_filtrado = df_grd_2023[mascara_respiratoria].copy()

ruta_salida = '../datos_grd/GRD_PUBLICO_2023_ER.txt'
grd_2023_filtrado.to_csv(
    ruta_salida,
    sep='|',
    index=False,
    encoding='Latin-1',
    lineterminator='\n'
)

print(f"Total de registros originales: {len(df_grd_2023):,}")
print(f"Registros retenidos (DIAGNOSTICO1 empieza con 'J'): {len(grd_2023_filtrado):,}")
print(f"Archivo exportado: {ruta_salida}")

display(grd_2023_filtrado[['CIP_ENCRIPTADO', 'DIAGNOSTICO1', 'FECHA_INGRESO', 'FECHAALTA']].head())

## 2022

In [16]:
df_grd_2022 = pd.read_csv('../datos_grd/GRD_PUBLICO_2022.txt', sep='|', encoding='utf-16', low_memory=False, on_bad_lines='skip')

mascara_respiratoria = df_grd_2022['DIAGNOSTICO1'].astype(str).str.strip().str.upper().str.startswith('J')

grd_2022_filtrado = df_grd_2022[mascara_respiratoria].copy()

ruta_salida = '../datos_grd/GRD_PUBLICO_2022_ER.txt'
grd_2022_filtrado.to_csv(
    ruta_salida,
    sep='|',
    index=False,
    encoding='Latin-1',
    errors = "replace",
    lineterminator='\n'
)

print(f"Total de registros originales: {len(df_grd_2022):,}")
print(f"Registros retenidos (DIAGNOSTICO1 empieza con 'J'): {len(grd_2022_filtrado):,}")
print(f"Archivo exportado: {ruta_salida}")

display(grd_2022_filtrado[['CIP_ENCRIPTADO', 'DIAGNOSTICO1', 'FECHA_INGRESO', 'FECHAALTA']].head())

Total de registros originales: 932,839
Registros retenidos (DIAGNOSTICO1 empieza con 'J'): 66,636
Archivo exportado: ../datos_grd/GRD_PUBLICO_2022_ER.txt


,CIP_ENCRIPTADO,DIAGNOSTICO1,FECHA_INGRESO,FECHAALTA
4,72715687,J40,2022-06-01,2022-07-04
13,97960221,J20.9,2022-01-28,2022-01-30
18,83466549,J21.8,2022-09-24,2022-09-26
48,70880670,J15.8,2022-10-11,2022-10-20
50,97417965,J20.9,2022-05-13,2022-05-15


## 2021

In [18]:
df_grd_2021 = pd.read_csv('../datos_grd/GRD_PUBLICO_2021.txt', sep='|', encoding='utf-8', low_memory=False)

mascara_respiratoria = df_grd_2021['DIAGNOSTICO1'].astype(str).str.strip().str.upper().str.startswith('J')

grd_2021_filtrado = df_grd_2021[mascara_respiratoria].copy()

ruta_salida = '../datos_grd/GRD_PUBLICO_2021_ER.txt'
grd_2021_filtrado.to_csv(
    ruta_salida,
    sep='|',
    index=False,
    encoding='Latin-1',
    lineterminator='\n'
)

print(f"Total de registros originales: {len(df_grd_2021):,}")
print(f"Registros retenidos (DIAGNOSTICO1 empieza con 'J'): {len(grd_2021_filtrado):,}")
print(f"Archivo exportado: {ruta_salida}")

display(grd_2021_filtrado[['CIP_ENCRIPTADO', 'DIAGNOSTICO1', 'FECHA_INGRESO', 'FECHAALTA']].head())

Total de registros originales: 816,909
Registros retenidos (DIAGNOSTICO1 empieza con 'J'): 34,618
Archivo exportado: ../datos_grd/GRD_PUBLICO_2021_ER.txt


,CIP_ENCRIPTADO,DIAGNOSTICO1,FECHA_INGRESO,FECHAALTA
2,74812628.0,J18.9,2021-10-19,2021-10-27
108,70907978.0,J44.1,2021-01-10,2021-01-11
146,69431542.0,J81,2021-01-20,2021-01-21
164,75135096.0,J12.8,2021-11-09,2021-11-12
175,68278038.0,J44.1,2021-04-10,2021-04-15


## 2020

In [3]:
df_grd_2020 = pd.read_csv('../datos_grd/GRD_PUBLICO_2020.txt', sep='|', encoding='utf-8', low_memory=False)

mascara_respiratoria = df_grd_2020['DIAGNOSTICO1'].astype(str).str.strip().str.upper().str.startswith('J')

grd_2020_filtrado = df_grd_2020[mascara_respiratoria].copy()

ruta_salida = '../datos_grd/GRD_PUBLICO_2020_ER.txt'
grd_2020_filtrado.to_csv(
    ruta_salida,
    sep='|',
    index=False,
    encoding='Latin-1',
    lineterminator='\n'
)

print(f"Total de registros originales: {len(df_grd_2020):,}")
print(f"Registros retenidos (DIAGNOSTICO1 empieza con 'J'): {len(grd_2020_filtrado):,}")
print(f"Archivo exportado: {ruta_salida}")

display(grd_2020_filtrado[['CIP_ENCRIPTADO', 'DIAGNOSTICO1', 'FECHA_INGRESO', 'FECHAALTA']].head())

Total de registros originales: 781,912
Registros retenidos (DIAGNOSTICO1 empieza con 'J'): 34,962
Archivo exportado: ../datos_grd/GRD_PUBLICO_2020_ER.txt


,CIP_ENCRIPTADO,DIAGNOSTICO1,FECHA_INGRESO,FECHAALTA
9,380332,J36,2020-12-02,2020-12-07
71,180907,J18.8,2020-05-06,2020-05-08
72,1418355,J12.8,2020-09-04,2020-09-08
116,514606,J45.9,2020-03-30,2020-04-01
173,892836,J34.2,2020-01-16,2020-01-16


## 2019

In [4]:
df_grd_2019 = pd.read_csv('../datos_grd/GRD_PUBLICO_2019.txt', sep='|', encoding='utf-8', low_memory=False)

mascara_respiratoria = df_grd_2019['DIAGNOSTICO1'].astype(str).str.strip().str.upper().str.startswith('J')

grd_2019_filtrado = df_grd_2019[mascara_respiratoria].copy()

ruta_salida = '../datos_grd/GRD_PUBLICO_2019_ER.txt'
grd_2019_filtrado.to_csv(
    ruta_salida,
    sep='|',
    index=False,
    encoding='Latin-1',
    lineterminator='\n'
)

print(f"Total de registros originales: {len(df_grd_2019):,}")
print(f"Registros retenidos (DIAGNOSTICO1 empieza con 'J'): {len(grd_2019_filtrado):,}")
print(f"Archivo exportado: {ruta_salida}")

display(grd_2019_filtrado[['CIP_ENCRIPTADO', 'DIAGNOSTICO1', 'FECHA_INGRESO', 'FECHAALTA']].head())

Total de registros originales: 1,151,475
Registros retenidos (DIAGNOSTICO1 empieza con 'J'): 96,439
Archivo exportado: ../datos_grd/GRD_PUBLICO_2019_ER.txt


,CIP_ENCRIPTADO,DIAGNOSTICO1,FECHA_INGRESO,FECHAALTA
6,1146635,J18.8,2019-02-05,2019-03-20
22,536006,J90,2019-03-06,2019-03-25
50,73162,J95.0,2019-03-04,2019-03-04
61,833468,J34.3,2019-04-02,2019-04-03
70,884097,J34.2,2019-04-01,2019-04-02
